# SafeFundus — Notebook 09: Fusion, Calibration, and Ablation

## Purpose

The SafeFundus pipeline now has two frozen open-set safety signals:

1. **Normal-Aware Dual-Mode JointEnergy**
2. **MC Dropout uncertainty**

This notebook performs the primary SafeFundus score-level fusion:

\[
R(x)
=
0.5J_N(x)
+
0.5U_N(x)
\]

where:

- \(J_N\) is the final validation-standardized Normal-Aware JointEnergy unknownness;
- \(U_N\) is the validation-standardized MC Dropout uncertainty from Notebook 08.

The fusion weight is **fixed**:

\[
\alpha=0.5
\]

and is **not tuned on the unknown test set**.

---

# Required ablation

This notebook compares:

1. **Classifier only**
2. **Normal-Aware JointEnergy only**
3. **MC Dropout only**
4. **JointEnergy + MC Dropout fusion**

The primary referral operating point remains:

\[
10\% \text{ familiar validation false-referral target}
\]

with 5% and 15% reported as sensitivity analysis.

---

# Temperature scaling

This notebook also fits one scalar temperature \(T\) per seed using the familiar validation logits only:

\[
p_c^{cal}
=
\sigma(z_c/T)
\]

Temperature scaling is used to improve the reliability of the **reported class probabilities**.

The open-set scores remain defined exactly as established in Notebooks 07B and 08:

- JointEnergy uses the frozen raw logits;
- MC Dropout uses the frozen stochastic probability variance.

This prevents calibration from silently changing the already-evaluated open-set score definitions.

For class decisions after calibration, the original validation-selected probability thresholds are transformed analytically so that the underlying binary decisions remain unchanged:

\[
\tau_c^{cal}
=
\sigma
\left(
\frac{\operatorname{logit}(\tau_c)}{T}
\right)
\]

Thus calibration changes probability reliability, not the established disease decision rule.

---

# No test-set tuning

Unknown-only test cases are used only to evaluate the frozen methods.

They are not used to choose:

- backbone;
- loss;
- temperature;
- score normalization;
- fusion weight;
- referral threshold;
- disease thresholds.


In [1]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
import json
import zipfile
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
)

from scipy.optimize import minimize_scalar

warnings.filterwarnings("ignore")

print("SafeFundus — Notebook 09")


SafeFundus — Notebook 09


# 2. Required Kaggle Inputs

Attach these two notebook outputs:

1. **`06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier`**
2. **`08_MC_Dropout_Uncertainty`**

The original RFMiD images are not required because this notebook works entirely from saved logits, labels, thresholds, and uncertainty scores.

Notebook 07B does not need to be attached; its Normal-Aware JointEnergy score is recomputed exactly from the 06C logits.


In [2]:
# ============================================================
# 2. Configuration
# ============================================================

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

SOURCE_06C = (
    "06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier"
)

SOURCE_08 = (
    "08_MC_Dropout_Uncertainty"
)

NOTEBOOK_STEM = (
    "09_SafeFundus_Fusion_Calibration_Ablation"
)

OUTPUT_DIR = (
    WORK_ROOT /
    NOTEBOOK_STEM
)

TABLE_DIR = (
    OUTPUT_DIR /
    "tables"
)

FIG_DIR = (
    OUTPUT_DIR /
    "figures"
)

PER_SEED_DIR = (
    OUTPUT_DIR /
    "per_seed"
)

for p in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIG_DIR,
    PER_SEED_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

CONFIG = {
    "seeds": [
        42,
        123,
        2026,
    ],

    "fusion_alpha_jointenergy":
        0.50,

    "fusion_alpha_mc_dropout":
        0.50,

    "primary_val_false_referral_rate":
        0.10,

    "sensitivity_val_false_referral_rates": [
        0.05,
        0.10,
        0.15,
    ],

    "normal_class":
        "NORMAL",

    "known_diseases": [
        "DR", "MH", "ODC", "TSLN",
        "DN", "MYA", "ARMD", "BRVO",
        "ODP", "ODE", "LS", "RS",
        "CSR", "CRS",
    ],

    "ece_bins":
        15,
}

NORMAL_CLASS = (
    CONFIG[
        "normal_class"
    ]
)

KNOWN_DISEASES = (
    CONFIG[
        "known_diseases"
    ]
)

MODEL_CLASSES = [
    NORMAL_CLASS,
    *KNOWN_DISEASES,
]

print(
    json.dumps(
        CONFIG,
        indent=2,
    )
)


{
  "seeds": [
    42,
    123,
    2026
  ],
  "fusion_alpha_jointenergy": 0.5,
  "fusion_alpha_mc_dropout": 0.5,
  "primary_val_false_referral_rate": 0.1,
  "sensitivity_val_false_referral_rates": [
    0.05,
    0.1,
    0.15
  ],
  "normal_class": "NORMAL",
  "known_diseases": [
    "DR",
    "MH",
    "ODC",
    "TSLN",
    "DN",
    "MYA",
    "ARMD",
    "BRVO",
    "ODP",
    "ODE",
    "LS",
    "RS",
    "CSR",
    "CRS"
  ],
  "ece_bins": 15
}


# 3. Locate / Extract the 06C and 08 Outputs


In [3]:
# ============================================================
# 3. Input locator
# ============================================================

def locate_or_extract(
    source_name,
    signature_file,
    work_extract_name,
):
    zip_candidates = list(
        INPUT_ROOT.rglob(
            f"{source_name}.zip"
        )
    )

    if zip_candidates:
        source_zip = (
            zip_candidates[0]
        )

        extract_dir = (
            WORK_ROOT /
            work_extract_name
        )

        if extract_dir.exists():
            shutil.rmtree(
                extract_dir
            )

        extract_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        with zipfile.ZipFile(
            source_zip,
            "r",
        ) as zf:
            zf.extractall(
                extract_dir
            )

        print(
            f"Found {source_name} ZIP:"
        )
        print(source_zip)

        return extract_dir

    candidates = list(
        INPUT_ROOT.rglob(
            signature_file
        )
    )

    for p in candidates:
        if (
            "seed_42"
            in str(p)
        ):
            # Expected:
            # root/seed_42/predictions/file
            # or root/per_seed/seed_42/file
            parts = p.parts

            if "per_seed" in parts:
                idx = (
                    parts.index(
                        "per_seed"
                    )
                )
                root = Path(
                    *parts[
                        :idx
                    ]
                )

            else:
                # root/seed_42/predictions/file
                root = p.parents[2]

            print(
                f"Found extracted {source_name}:"
            )
            print(root)

            return root

    raise FileNotFoundError(
        f"Could not find {source_name}. "
        f"Attach its Kaggle output."
    )


ROOT_06C = locate_or_extract(
    SOURCE_06C,
    "validation_logits_predictions_15class.csv",
    "notebook06c_for_fusion",
)

ROOT_08 = locate_or_extract(
    SOURCE_08,
    "validation_mc_dropout_scores.csv",
    "notebook08_for_fusion",
)

print("\n06C root:")
print(ROOT_06C)

print("\n08 root:")
print(ROOT_08)


Found 06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier ZIP:
/kaggle/input/notebooks/shouravdebnath/06c-swintiny-asl-normalaware-multiseed-classifier/06C_SwinTiny_ASL_NormalAware_MultiSeed_Classifier.zip
Found 08_MC_Dropout_Uncertainty ZIP:
/kaggle/input/notebooks/shouravdebnath/08-mc-dropout-uncertainty/08_MC_Dropout_Uncertainty.zip

06C root:
/kaggle/working/notebook06c_for_fusion

08 root:
/kaggle/working/notebook08_for_fusion


# 4. Load Per-Seed Inputs


In [4]:
# ============================================================
# 4. Data loader
# ============================================================

def load_seed_data(seed):

    seed06 = (
        ROOT_06C /
        f"seed_{seed}"
    )

    pred06 = (
        seed06 /
        "predictions"
    )

    table06 = (
        seed06 /
        "tables"
    )

    mc_seed = (
        ROOT_08 /
        "per_seed" /
        f"seed_{seed}"
    )

    if not mc_seed.exists():
        # Support extracted output where per_seed
        # is itself the attached root.
        alternate = (
            ROOT_08 /
            f"seed_{seed}"
        )

        if alternate.exists():
            mc_seed = alternate

    paths = {
        "validation":
            pred06 /
            "validation_logits_predictions_15class.csv",

        "familiar":
            pred06 /
            "familiar_test_logits_predictions_15class.csv",

        "unknown":
            pred06 /
            "unknown_test_logits_predictions_15class.csv",

        "mixed":
            pred06 /
            "mixed_test_logits_predictions_15class.csv",

        "intermediate":
            pred06 /
            "intermediate_test_logits_predictions_15class.csv",

        "thresholds":
            table06 /
            "validation_thresholds_15class.csv",

        "mc_validation":
            mc_seed /
            "validation_mc_dropout_scores.csv",

        "mc_familiar":
            mc_seed /
            "familiar_mc_dropout_scores.csv",

        "mc_unknown":
            mc_seed /
            "unknown_mc_dropout_scores.csv",

        "mc_mixed":
            mc_seed /
            "mixed_mc_dropout_scores.csv",

        "mc_intermediate":
            mc_seed /
            "intermediate_mc_dropout_scores.csv",
    }

    missing = [
        str(p)
        for p in paths.values()
        if not p.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Missing required files:\n"
            + "\n".join(
                missing
            )
        )

    result = {
        key: pd.read_csv(
            path
        )
        for key, path
        in paths.items()
    }

    threshold_map = dict(
        zip(
            result[
                "thresholds"
            ][
                "Class"
            ],
            result[
                "thresholds"
            ][
                "Threshold"
            ],
        )
    )

    result[
        "threshold_map"
    ] = threshold_map

    # Strict ID alignment checks.
    pairs = [
        (
            "validation",
            "mc_validation",
        ),
        (
            "familiar",
            "mc_familiar",
        ),
        (
            "unknown",
            "mc_unknown",
        ),
        (
            "mixed",
            "mc_mixed",
        ),
        (
            "intermediate",
            "mc_intermediate",
        ),
    ]

    for a, b in pairs:
        id_a = (
            result[a]["ID"]
            .astype(str)
            .tolist()
        )

        id_b = (
            result[b]["ID"]
            .astype(str)
            .tolist()
        )

        if id_a != id_b:
            raise ValueError(
                f"ID/order mismatch: "
                f"{a} vs {b}"
            )

    return result


for seed in CONFIG["seeds"]:
    data = load_seed_data(
        seed
    )

    print(
        f"Seed {seed}: "
        f"val={len(data['validation'])}, "
        f"familiar={len(data['familiar'])}, "
        f"unknown={len(data['unknown'])}, "
        f"mixed={len(data['mixed'])}, "
        f"intermediate={len(data['intermediate'])}"
    )


Seed 42: val=560, familiar=571, unknown=32, mixed=24, intermediate=13
Seed 123: val=560, familiar=571, unknown=32, mixed=24, intermediate=13
Seed 2026: val=560, familiar=571, unknown=32, mixed=24, intermediate=13


# 5. Recompute Normal-Aware Dual-Mode JointEnergy

This is the exact score from Notebook 07B.

Validation Normal images fit the normal branch.

Validation Known_Only images fit the disease branch.

At inference:

\[
J_{NA}
=
-\max(Z_N,Z_D)
\]

We then perform one additional **full-validation z-standardization**:

\[
J_N
=
\frac{J_{NA}-\mu_{JNA}}
{\sigma_{JNA}}
\]

This final standardization puts JointEnergy on the same validation z-score scale as \(U_N\) before the fixed 50/50 fusion.


In [5]:
# ============================================================
# 5. JointEnergy
# ============================================================

def softplus_np(x):
    return np.logaddexp(
        0.0,
        x,
    )


def safe_std(x):
    value = float(
        np.std(
            x,
            ddof=0,
        )
    )

    if value < 1e-12:
        raise ValueError(
            "Standard deviation is too small."
        )

    return value


def energy_branches(df):

    normal_evidence = (
        softplus_np(
            df[
                "logit_NORMAL"
            ].to_numpy(
                dtype=float
            )
        )
    )

    disease_logits = (
        df[
            [
                f"logit_{c}"
                for c in KNOWN_DISEASES
            ]
        ]
        .to_numpy(
            dtype=float
        )
    )

    disease_evidence = (
        softplus_np(
            disease_logits
        )
        .sum(axis=1)
    )

    return (
        normal_evidence,
        disease_evidence,
    )


def fit_normalaware_jointenergy(
    validation_df,
):
    normal_evidence, disease_evidence = (
        energy_branches(
            validation_df
        )
    )

    groups = (
        validation_df[
            "Group"
        ]
        .astype(str)
        .to_numpy()
    )

    normal_mask = (
        groups
        == "Normal"
    )

    known_mask = (
        groups
        == "Known_Only"
    )

    params = {
        "mu_normal":
            float(
                normal_evidence[
                    normal_mask
                ].mean()
            ),

        "sd_normal":
            safe_std(
                normal_evidence[
                    normal_mask
                ]
            ),

        "mu_disease":
            float(
                disease_evidence[
                    known_mask
                ].mean()
            ),

        "sd_disease":
            safe_std(
                disease_evidence[
                    known_mask
                ]
            ),
    }

    return params


def apply_normalaware_jointenergy(
    df,
    params,
):
    normal_evidence, disease_evidence = (
        energy_branches(
            df
        )
    )

    z_normal = (
        normal_evidence
        - params[
            "mu_normal"
        ]
    ) / params[
        "sd_normal"
    ]

    z_disease = (
        disease_evidence
        - params[
            "mu_disease"
        ]
    ) / params[
        "sd_disease"
    ]

    familiar_evidence = (
        np.maximum(
            z_normal,
            z_disease,
        )
    )

    return (
        -familiar_evidence
    )


# 6. Temperature Scaling

A single scalar temperature is fit independently for each seed on all 15 familiar validation outputs by minimizing BCE.

This calibration is **not** used to tune or redefine the open-set scores.


In [6]:
# ============================================================
# 6. Temperature scaling
# ============================================================

def sigmoid_np(x):
    # Stable enough for the observed logits.
    return (
        1.0 /
        (
            1.0
            + np.exp(
                -np.clip(
                    x,
                    -60,
                    60,
                )
            )
        )
    )


def stable_bce_with_logits(
    logits,
    targets,
):
    loss = (
        np.maximum(
            logits,
            0.0,
        )
        - logits * targets
        + np.log1p(
            np.exp(
                -np.abs(
                    logits
                )
            )
        )
    )

    return float(
        loss.mean()
    )


def fit_temperature(
    validation_df,
):
    logits = np.stack(
        [
            validation_df[
                f"logit_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in MODEL_CLASSES
        ],
        axis=1,
    )

    targets = np.stack(
        [
            validation_df[
                f"true_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in MODEL_CLASSES
        ],
        axis=1,
    )

    def objective(
        log_temperature,
    ):
        temperature = np.exp(
            log_temperature
        )

        return stable_bce_with_logits(
            logits /
            temperature,
            targets,
        )

    result = minimize_scalar(
        objective,
        bounds=(-3.0, 3.0),
        method="bounded",
    )

    temperature = float(
        np.exp(
            result.x
        )
    )

    return temperature


def calibrated_probabilities(
    df,
    temperature,
):
    logits = np.stack(
        [
            df[
                f"logit_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in MODEL_CLASSES
        ],
        axis=1,
    )

    return sigmoid_np(
        logits /
        temperature
    )


def transform_threshold(
    threshold,
    temperature,
    eps=1e-8,
):
    t = float(
        np.clip(
            threshold,
            eps,
            1.0 - eps,
        )
    )

    logit_t = np.log(
        t /
        (1.0 - t)
    )

    return float(
        sigmoid_np(
            logit_t /
            temperature
        )
    )


def brier_score_multilabel(
    targets,
    probs,
):
    return float(
        np.mean(
            (
                probs
                - targets
            ) ** 2
        )
    )


def ece_multilabel(
    targets,
    probs,
    n_bins=15,
):
    # Pool all binary label predictions.
    y = targets.reshape(-1)
    p = probs.reshape(-1)

    bins = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    ece = 0.0

    for i in range(
        n_bins
    ):
        lo = bins[i]
        hi = bins[i + 1]

        if i == (
            n_bins - 1
        ):
            mask = (
                (p >= lo)
                & (p <= hi)
            )
        else:
            mask = (
                (p >= lo)
                & (p < hi)
            )

        if not mask.any():
            continue

        confidence = float(
            p[mask].mean()
        )

        accuracy = float(
            y[mask].mean()
        )

        weight = float(
            mask.mean()
        )

        ece += (
            weight
            * abs(
                accuracy
                - confidence
            )
        )

    return float(ece)


# 7. Unknown Detection and Safety Helpers


In [7]:
# ============================================================
# 7. Metric helpers
# ============================================================

def fpr_at_95_tpr(
    y_true,
    scores,
):
    fpr, tpr, thresholds = (
        roc_curve(
            y_true,
            scores,
        )
    )

    valid = np.where(
        tpr >= 0.95
    )[0]

    if len(valid) == 0:
        return np.nan

    return float(
        np.min(
            fpr[valid]
        )
    )


def ood_metrics(
    familiar_scores,
    unknown_scores,
):
    y_true = np.concatenate([
        np.zeros(
            len(
                familiar_scores
            ),
            dtype=int,
        ),
        np.ones(
            len(
                unknown_scores
            ),
            dtype=int,
        ),
    ])

    scores = np.concatenate([
        familiar_scores,
        unknown_scores,
    ])

    return {
        "unknown_auroc":
            float(
                roc_auc_score(
                    y_true,
                    scores,
                )
            ),

        "unknown_aupr":
            float(
                average_precision_score(
                    y_true,
                    scores,
                )
            ),

        "fpr_at_95_tpr":
            fpr_at_95_tpr(
                y_true,
                scores,
            ),
    }


def closedset_unknown_decisions(
    unknown_df,
    threshold_map,
):
    disease_prob = np.stack(
        [
            unknown_df[
                f"prob_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in KNOWN_DISEASES
        ],
        axis=1,
    )

    disease_thresholds = np.asarray(
        [
            threshold_map[c]
            for c in KNOWN_DISEASES
        ],
        dtype=float,
    )

    disease_pred = (
        disease_prob
        >= disease_thresholds[
            None, :
        ]
    )

    any_disease = (
        disease_pred
        .any(axis=1)
    )

    no_disease = (
        ~any_disease
    )

    normal_prob = (
        unknown_df[
            "prob_NORMAL"
        ].to_numpy(
            dtype=float
        )
    )

    explicit_normal = (
        normal_prob
        >= float(
            threshold_map[
                "NORMAL"
            ]
        )
    )

    return {
        "any_disease":
            any_disease,
        "no_disease":
            no_disease,
        "explicit_normal":
            explicit_normal,
    }


def evaluate_referral_method(
    method_name,
    val_scores,
    familiar_scores,
    unknown_scores,
    mixed_scores,
    intermediate_scores,
    familiar_groups,
    decisions,
    target_frr,
):
    threshold = float(
        np.quantile(
            val_scores,
            1.0 - target_frr,
        )
    )

    val_ref = (
        val_scores
        > threshold
    )

    familiar_ref = (
        familiar_scores
        > threshold
    )

    unknown_ref = (
        unknown_scores
        > threshold
    )

    mixed_ref = (
        mixed_scores
        > threshold
    )

    intermediate_ref = (
        intermediate_scores
        > threshold
    )

    normal_mask = (
        familiar_groups
        == "Normal"
    )

    known_mask = (
        familiar_groups
        == "Known_Only"
    )

    residual = (
        ~unknown_ref
    )

    rare_to_normal = (
        residual
        & decisions[
            "no_disease"
        ]
    )

    strict_false_reassurance = (
        rare_to_normal
        & decisions[
            "explicit_normal"
        ]
    )

    forced_known = (
        residual
        & decisions[
            "any_disease"
        ]
    )

    row = {
        "method":
            method_name,

        "threshold":
            threshold,

        "actual_val_false_referral_rate":
            float(
                val_ref.mean()
            ),

        "familiar_test_false_referral_rate":
            float(
                familiar_ref.mean()
            ),

        "normal_test_false_referral_rate":
            float(
                familiar_ref[
                    normal_mask
                ].mean()
            ),

        "known_only_test_false_referral_rate":
            float(
                familiar_ref[
                    known_mask
                ].mean()
            ),

        "unknown_referral_sensitivity":
            float(
                unknown_ref.mean()
            ),

        "mixed_referral_rate":
            float(
                mixed_ref.mean()
            ),

        "intermediate_referral_rate":
            float(
                intermediate_ref.mean()
            ),

        "post_referral_rare_to_normal_rate":
            float(
                rare_to_normal.mean()
            ),

        "post_referral_explicit_normal_false_reassurance":
            float(
                strict_false_reassurance.mean()
            ),

        "post_referral_forced_known_rate":
            float(
                forced_known.mean()
            ),
    }

    row.update(
        ood_metrics(
            familiar_scores,
            unknown_scores,
        )
    )

    return (
        row,
        {
            "threshold":
                threshold,
            "val_referral":
                val_ref,
            "familiar_referral":
                familiar_ref,
            "unknown_referral":
                unknown_ref,
        },
    )


# 8. Run the Full Three-Seed Fusion Experiment

The fusion weights are fixed at:

\[
0.5 / 0.5
\]

No alternative weight is searched on the test data.


In [8]:
# ============================================================
# 8. Main experiment
# ============================================================

all_method_rows = []
calibration_rows = []
overlap_rows = []
fusion_sensitivity_rows = []

for seed in CONFIG["seeds"]:

    print("\n" + "#" * 80)
    print(
        f"SAFEFUNDUS FUSION — SEED {seed}"
    )
    print("#" * 80)

    data = load_seed_data(
        seed
    )

    val_df = data[
        "validation"
    ]

    familiar_df = data[
        "familiar"
    ]

    unknown_df = data[
        "unknown"
    ]

    mixed_df = data[
        "mixed"
    ]

    intermediate_df = data[
        "intermediate"
    ]

    threshold_map = data[
        "threshold_map"
    ]

    # ========================================================
    # JointEnergy
    # ========================================================

    joint_params = (
        fit_normalaware_jointenergy(
            val_df
        )
    )

    j_val_raw = (
        apply_normalaware_jointenergy(
            val_df,
            joint_params,
        )
    )

    j_familiar_raw = (
        apply_normalaware_jointenergy(
            familiar_df,
            joint_params,
        )
    )

    j_unknown_raw = (
        apply_normalaware_jointenergy(
            unknown_df,
            joint_params,
        )
    )

    j_mixed_raw = (
        apply_normalaware_jointenergy(
            mixed_df,
            joint_params,
        )
    )

    j_intermediate_raw = (
        apply_normalaware_jointenergy(
            intermediate_df,
            joint_params,
        )
    )

    joint_final_mu = float(
        j_val_raw.mean()
    )

    joint_final_sd = safe_std(
        j_val_raw
    )

    def normalize_joint(x):
        return (
            x
            - joint_final_mu
        ) / joint_final_sd

    j_val = normalize_joint(
        j_val_raw
    )

    j_familiar = normalize_joint(
        j_familiar_raw
    )

    j_unknown = normalize_joint(
        j_unknown_raw
    )

    j_mixed = normalize_joint(
        j_mixed_raw
    )

    j_intermediate = normalize_joint(
        j_intermediate_raw
    )

    # ========================================================
    # MC Dropout
    # ========================================================

    u_val = data[
        "mc_validation"
    ][
        "U_MC_Normalized"
    ].to_numpy(
        dtype=float
    )

    u_familiar = data[
        "mc_familiar"
    ][
        "U_MC_Normalized"
    ].to_numpy(
        dtype=float
    )

    u_unknown = data[
        "mc_unknown"
    ][
        "U_MC_Normalized"
    ].to_numpy(
        dtype=float
    )

    u_mixed = data[
        "mc_mixed"
    ][
        "U_MC_Normalized"
    ].to_numpy(
        dtype=float
    )

    u_intermediate = data[
        "mc_intermediate"
    ][
        "U_MC_Normalized"
    ].to_numpy(
        dtype=float
    )

    # ========================================================
    # Fixed 50/50 fusion
    # ========================================================

    alpha_j = CONFIG[
        "fusion_alpha_jointenergy"
    ]

    alpha_u = CONFIG[
        "fusion_alpha_mc_dropout"
    ]

    r_val = (
        alpha_j * j_val
        + alpha_u * u_val
    )

    r_familiar = (
        alpha_j * j_familiar
        + alpha_u * u_familiar
    )

    r_unknown = (
        alpha_j * j_unknown
        + alpha_u * u_unknown
    )

    r_mixed = (
        alpha_j * j_mixed
        + alpha_u * u_mixed
    )

    r_intermediate = (
        alpha_j * j_intermediate
        + alpha_u * u_intermediate
    )

    # ========================================================
    # Closed-set decisions
    # ========================================================

    decisions = (
        closedset_unknown_decisions(
            unknown_df,
            threshold_map,
        )
    )

    familiar_groups = (
        familiar_df[
            "Group"
        ]
        .astype(str)
        .to_numpy()
    )

    # ========================================================
    # Ablation methods
    # ========================================================

    target_frr = CONFIG[
        "primary_val_false_referral_rate"
    ]

    method_specs = {
        "JointEnergy":
            (
                j_val,
                j_familiar,
                j_unknown,
                j_mixed,
                j_intermediate,
            ),

        "MC_Dropout":
            (
                u_val,
                u_familiar,
                u_unknown,
                u_mixed,
                u_intermediate,
            ),

        "Fusion_50_50":
            (
                r_val,
                r_familiar,
                r_unknown,
                r_mixed,
                r_intermediate,
            ),
    }

    referral_masks = {}

    for method_name, scores in (
        method_specs.items()
    ):
        row, masks = (
            evaluate_referral_method(
                method_name,
                *scores,
                familiar_groups,
                decisions,
                target_frr,
            )
        )

        row[
            "seed"
        ] = seed

        row[
            "closedset_rare_to_normal_rate"
        ] = float(
            decisions[
                "no_disease"
            ].mean()
        )

        all_method_rows.append(
            row
        )

        referral_masks[
            method_name
        ] = masks

    # Classifier-only baseline.
    classifier_row = {
        "method":
            "Classifier_Only",

        "threshold":
            np.nan,

        "actual_val_false_referral_rate":
            0.0,

        "familiar_test_false_referral_rate":
            0.0,

        "normal_test_false_referral_rate":
            0.0,

        "known_only_test_false_referral_rate":
            0.0,

        "unknown_referral_sensitivity":
            0.0,

        "mixed_referral_rate":
            0.0,

        "intermediate_referral_rate":
            0.0,

        "closedset_rare_to_normal_rate":
            float(
                decisions[
                    "no_disease"
                ].mean()
            ),

        "post_referral_rare_to_normal_rate":
            float(
                decisions[
                    "no_disease"
                ].mean()
            ),

        "post_referral_explicit_normal_false_reassurance":
            float(
                (
                    decisions[
                        "no_disease"
                    ]
                    & decisions[
                        "explicit_normal"
                    ]
                ).mean()
            ),

        "post_referral_forced_known_rate":
            float(
                decisions[
                    "any_disease"
                ].mean()
            ),

        "unknown_auroc":
            np.nan,

        "unknown_aupr":
            np.nan,

        "fpr_at_95_tpr":
            np.nan,

        "seed":
            seed,
    }

    all_method_rows.append(
        classifier_row
    )

    # ========================================================
    # Complementarity / referral overlap
    # ========================================================

    j_unknown_ref = (
        referral_masks[
            "JointEnergy"
        ][
            "unknown_referral"
        ]
    )

    u_unknown_ref = (
        referral_masks[
            "MC_Dropout"
        ][
            "unknown_referral"
        ]
    )

    fusion_unknown_ref = (
        referral_masks[
            "Fusion_50_50"
        ][
            "unknown_referral"
        ]
    )

    overlap_rows.append({
        "seed":
            seed,

        "unknown_cases":
            int(
                len(
                    unknown_df
                )
            ),

        "jointenergy_referred":
            int(
                j_unknown_ref.sum()
            ),

        "mc_dropout_referred":
            int(
                u_unknown_ref.sum()
            ),

        "jointenergy_and_mc_intersection":
            int(
                (
                    j_unknown_ref
                    & u_unknown_ref
                ).sum()
            ),

        "jointenergy_only":
            int(
                (
                    j_unknown_ref
                    & ~u_unknown_ref
                ).sum()
            ),

        "mc_dropout_only":
            int(
                (
                    u_unknown_ref
                    & ~j_unknown_ref
                ).sum()
            ),

        "union_joint_or_mc":
            int(
                (
                    j_unknown_ref
                    | u_unknown_ref
                ).sum()
            ),

        "fusion_referred":
            int(
                fusion_unknown_ref.sum()
            ),
    })

    # ========================================================
    # Fusion threshold sensitivity
    # ========================================================

    normal_mask = (
        familiar_groups
        == "Normal"
    )

    known_mask = (
        familiar_groups
        == "Known_Only"
    )

    for requested_frr in CONFIG[
        "sensitivity_val_false_referral_rates"
    ]:

        threshold = float(
            np.quantile(
                r_val,
                1.0
                - requested_frr,
            )
        )

        familiar_ref = (
            r_familiar
            > threshold
        )

        unknown_ref = (
            r_unknown
            > threshold
        )

        residual = (
            ~unknown_ref
        )

        rare = (
            residual
            & decisions[
                "no_disease"
            ]
        )

        strict = (
            rare
            & decisions[
                "explicit_normal"
            ]
        )

        fusion_sensitivity_rows.append({
            "seed":
                seed,

            "requested_val_false_referral_rate":
                requested_frr,

            "threshold":
                threshold,

            "actual_val_false_referral_rate":
                float(
                    (
                        r_val
                        > threshold
                    ).mean()
                ),

            "familiar_test_false_referral_rate":
                float(
                    familiar_ref.mean()
                ),

            "normal_test_false_referral_rate":
                float(
                    familiar_ref[
                        normal_mask
                    ].mean()
                ),

            "known_only_test_false_referral_rate":
                float(
                    familiar_ref[
                        known_mask
                    ].mean()
                ),

            "unknown_referral_sensitivity":
                float(
                    unknown_ref.mean()
                ),

            "post_referral_rare_to_normal_rate":
                float(
                    rare.mean()
                ),

            "post_referral_explicit_normal_false_reassurance":
                float(
                    strict.mean()
                ),
        })

    # ========================================================
    # Temperature scaling / calibration
    # ========================================================

    temperature = (
        fit_temperature(
            val_df
        )
    )

    val_targets = np.stack(
        [
            val_df[
                f"true_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in MODEL_CLASSES
        ],
        axis=1,
    )

    familiar_targets = np.stack(
        [
            familiar_df[
                f"true_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in MODEL_CLASSES
        ],
        axis=1,
    )

    val_uncal = np.stack(
        [
            val_df[
                f"prob_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in MODEL_CLASSES
        ],
        axis=1,
    )

    familiar_uncal = np.stack(
        [
            familiar_df[
                f"prob_{c}"
            ].to_numpy(
                dtype=float
            )
            for c in MODEL_CLASSES
        ],
        axis=1,
    )

    val_cal = calibrated_probabilities(
        val_df,
        temperature,
    )

    familiar_cal = calibrated_probabilities(
        familiar_df,
        temperature,
    )

    transformed_thresholds = {
        class_name:
            transform_threshold(
                threshold_map[
                    class_name
                ],
                temperature,
            )
        for class_name
        in MODEL_CLASSES
    }

    calibration_rows.append({
        "seed":
            seed,

        "temperature":
            temperature,

        "validation_ece_before":
            ece_multilabel(
                val_targets,
                val_uncal,
                CONFIG[
                    "ece_bins"
                ],
            ),

        "validation_ece_after":
            ece_multilabel(
                val_targets,
                val_cal,
                CONFIG[
                    "ece_bins"
                ],
            ),

        "validation_brier_before":
            brier_score_multilabel(
                val_targets,
                val_uncal,
            ),

        "validation_brier_after":
            brier_score_multilabel(
                val_targets,
                val_cal,
            ),

        "familiar_test_ece_before":
            ece_multilabel(
                familiar_targets,
                familiar_uncal,
                CONFIG[
                    "ece_bins"
                ],
            ),

        "familiar_test_ece_after":
            ece_multilabel(
                familiar_targets,
                familiar_cal,
                CONFIG[
                    "ece_bins"
                ],
            ),

        "familiar_test_brier_before":
            brier_score_multilabel(
                familiar_targets,
                familiar_uncal,
            ),

        "familiar_test_brier_after":
            brier_score_multilabel(
                familiar_targets,
                familiar_cal,
            ),
    })

    # ========================================================
    # Save per-image final scores
    # ========================================================

    seed_out = (
        PER_SEED_DIR /
        f"seed_{seed}"
    )

    seed_out.mkdir(
        parents=True,
        exist_ok=True,
    )

    score_sets = {
        "validation":
            (
                val_df,
                j_val,
                u_val,
                r_val,
            ),

        "familiar":
            (
                familiar_df,
                j_familiar,
                u_familiar,
                r_familiar,
            ),

        "unknown":
            (
                unknown_df,
                j_unknown,
                u_unknown,
                r_unknown,
            ),

        "mixed":
            (
                mixed_df,
                j_mixed,
                u_mixed,
                r_mixed,
            ),

        "intermediate":
            (
                intermediate_df,
                j_intermediate,
                u_intermediate,
                r_intermediate,
            ),
    }

    fusion_threshold = (
        referral_masks[
            "Fusion_50_50"
        ][
            "threshold"
        ]
    )

    for split_name, (
        base_df,
        joint_score,
        mc_score,
        fusion_score,
    ) in score_sets.items():

        export_df = pd.DataFrame({
            "ID":
                base_df[
                    "ID"
                ],

            "Group":
                base_df[
                    "Group"
                ],

            "JointEnergy_Normalized":
                joint_score,

            "MC_Dropout_Normalized":
                mc_score,

            "Fusion_R":
                fusion_score,

            "Fusion_Referral":
                (
                    fusion_score
                    > fusion_threshold
                ).astype(int),
        })

        export_df.to_csv(
            seed_out /
            f"{split_name}_final_scores.csv",
            index=False,
        )

    with open(
        seed_out /
        "fusion_validation_parameters.json",
        "w",
    ) as f:
        json.dump(
            {
                "jointenergy_branch_parameters":
                    joint_params,

                "jointenergy_final_validation_mean":
                    joint_final_mu,

                "jointenergy_final_validation_sd":
                    joint_final_sd,

                "fusion_alpha_jointenergy":
                    alpha_j,

                "fusion_alpha_mc_dropout":
                    alpha_u,

                "fusion_primary_threshold":
                    float(
                        fusion_threshold
                    ),

                "temperature":
                    temperature,

                "calibrated_thresholds":
                    transformed_thresholds,
            },
            f,
            indent=2,
        )

    # ========================================================
    # Distribution plot
    # ========================================================

    plt.figure(
        figsize=(8, 5)
    )

    plt.hist(
        r_familiar,
        bins=30,
        density=True,
        alpha=0.6,
        label="Familiar test",
    )

    plt.hist(
        r_unknown,
        bins=15,
        density=True,
        alpha=0.6,
        label="Unknown-only test",
    )

    plt.axvline(
        fusion_threshold,
        linestyle="--",
        label="Validation fusion threshold",
    )

    plt.xlabel(
        "SafeFundus Fusion Referral Score R(x)"
    )

    plt.ylabel(
        "Density"
    )

    plt.title(
        f"SafeFundus Fusion — Seed {seed}"
    )

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIG_DIR /
        f"fusion_distribution_seed_{seed}.png",
        dpi=200,
        bbox_inches="tight",
    )

    plt.close()

print("\nAll three fusion seeds completed.")



################################################################################
SAFEFUNDUS FUSION — SEED 42
################################################################################

################################################################################
SAFEFUNDUS FUSION — SEED 123
################################################################################

################################################################################
SAFEFUNDUS FUSION — SEED 2026
################################################################################

All three fusion seeds completed.


# 9. Ablation Table — Mean ± SD Across Seeds


In [9]:
# ============================================================
# 9. Ablation aggregation
# ============================================================

method_results_df = pd.DataFrame(
    all_method_rows
)

method_results_df.to_csv(
    TABLE_DIR /
    "ablation_results_by_seed.csv",
    index=False,
)

display(
    method_results_df
)

ABLATION_METRICS = [
    "familiar_test_false_referral_rate",
    "normal_test_false_referral_rate",
    "known_only_test_false_referral_rate",
    "unknown_referral_sensitivity",
    "mixed_referral_rate",
    "intermediate_referral_rate",
    "closedset_rare_to_normal_rate",
    "post_referral_rare_to_normal_rate",
    "post_referral_explicit_normal_false_reassurance",
    "post_referral_forced_known_rate",
    "unknown_auroc",
    "unknown_aupr",
    "fpr_at_95_tpr",
]

aggregate_rows = []

METHOD_ORDER = [
    "Classifier_Only",
    "JointEnergy",
    "MC_Dropout",
    "Fusion_50_50",
]

for method_name in (
    METHOD_ORDER
):
    subset = (
        method_results_df[
            method_results_df[
                "method"
            ] == method_name
        ]
    )

    for metric in (
        ABLATION_METRICS
    ):
        values = pd.to_numeric(
            subset[
                metric
            ],
            errors="coerce",
        )

        aggregate_rows.append({
            "Method":
                method_name,

            "Metric":
                metric,

            "Mean":
                float(
                    values.mean()
                )
                if values.notna().any()
                else np.nan,

            "SD":
                float(
                    values.std(
                        ddof=1
                    )
                )
                if (
                    values.notna().sum()
                    > 1
                )
                else np.nan,
        })

ablation_mean_sd = pd.DataFrame(
    aggregate_rows
)

ablation_mean_sd.to_csv(
    TABLE_DIR /
    "ablation_mean_sd.csv",
    index=False,
)

display(
    ablation_mean_sd
)

print("\n" + "=" * 80)
print(
    "FINAL SAFEFUNDUS FUSION SUMMARY"
)
print("=" * 80)

fusion_summary = (
    ablation_mean_sd[
        ablation_mean_sd[
            "Method"
        ] == "Fusion_50_50"
    ]
)

for _, row in (
    fusion_summary
    .iterrows()
):
    print(
        f"{row['Metric']:55s}: "
        f"{row['Mean']:.6f} "
        f"± {row['SD']:.6f}"
    )


,method,threshold,actual_val_false_referral_rate,familiar_test_false_referral_rate,normal_test_false_referral_rate,known_only_test_false_referral_rate,unknown_referral_sensitivity,mixed_referral_rate,intermediate_referral_rate,post_referral_rare_to_normal_rate,post_referral_explicit_normal_false_reassurance,post_referral_forced_known_rate,unknown_auroc,unknown_aupr,fpr_at_95_tpr,seed,closedset_rare_to_normal_rate
0,JointEnergy,1.267278,0.1,0.094571,0.044776,0.109840,0.28125,0.166667,0.076923,0.18750,0.09375,0.53125,0.734895,0.136699,0.767075,42,0.21875
1,MC_Dropout,1.301160,0.1,0.108581,0.000000,0.141876,0.18750,0.166667,0.384615,0.21875,0.09375,0.59375,0.728930,0.135668,0.819615,42,0.21875
2,Fusion_50_50,0.862129,0.1,0.103327,0.007463,0.132723,0.50000,0.250000,0.307692,0.15625,0.09375,0.34375,0.799694,0.208246,0.616462,42,0.21875
3,Classifier_Only,NaN,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.21875,0.09375,0.78125,NaN,NaN,NaN,42,0.21875
4,JointEnergy,1.386004,0.1,0.061296,0.022388,0.073227,0.25000,0.083333,0.153846,0.15625,0.09375,0.59375,0.768717,0.198836,0.751313,123,0.28125
5,MC_Dropout,1.315755,0.1,0.064799,0.000000,0.084668,0.21875,0.333333,0.153846,0.12500,0.12500,0.65625,0.661066,0.128034,0.947461,123,0.28125
6,Fusion_50_50,0.955950,0.1,0.073555,0.014925,0.091533,0.37500,0.291667,0.384615,0.09375,0.09375,0.53125,0.758319,0.198188,0.819615,123,0.28125
7,Classifier_Only,NaN,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.28125,0.18750,0.71875,NaN,NaN,NaN,123,0.28125
8,JointEnergy,1.434464,0.1,0.084063,0.082090,0.084668,0.21875,0.083333,0.153846,0.18750,0.15625,0.59375,0.719626,0.137214,0.800350,2026,0.31250
9,MC_Dropout,1.335870,0.1,0.078809,0.000000,0.102975,0.37500,0.291667,0.461538,0.21875,0.18750,0.40625,0.704302,0.202727,0.817863,2026,0.31250


,Method,Metric,Mean,SD
0,Classifier_Only,familiar_test_false_referral_rate,0.000000,0.000000
1,Classifier_Only,normal_test_false_referral_rate,0.000000,0.000000
2,Classifier_Only,known_only_test_false_referral_rate,0.000000,0.000000
3,Classifier_Only,unknown_referral_sensitivity,0.000000,0.000000
4,Classifier_Only,mixed_referral_rate,0.000000,0.000000
5,Classifier_Only,intermediate_referral_rate,0.000000,0.000000
6,Classifier_Only,closedset_rare_to_normal_rate,0.270833,0.047735
7,Classifier_Only,post_referral_rare_to_normal_rate,0.270833,0.047735
8,Classifier_Only,post_referral_explicit_normal_false_reassurance,0.166667,0.065052
9,Classifier_Only,post_referral_forced_known_rate,0.729167,0.047735



FINAL SAFEFUNDUS FUSION SUMMARY
familiar_test_false_referral_rate                      : 0.094571 ± 0.018284
normal_test_false_referral_rate                        : 0.017413 ± 0.011399
known_only_test_false_referral_rate                    : 0.118230 ± 0.023149
unknown_referral_sensitivity                           : 0.479167 ± 0.095470
mixed_referral_rate                                    : 0.291667 ± 0.041667
intermediate_referral_rate                             : 0.384615 ± 0.076923
closedset_rare_to_normal_rate                          : 0.270833 ± 0.047735
post_referral_rare_to_normal_rate                      : 0.135417 ± 0.036084
post_referral_explicit_normal_false_reassurance        : 0.114583 ± 0.036084
post_referral_forced_known_rate                        : 0.385417 ± 0.130104
unknown_auroc                                          : 0.782144 ± 0.021389
unknown_aupr                                           : 0.219180 ± 0.028103
fpr_at_95_tpr                              

# 10. JointEnergy / MC Dropout Complementarity

This table quantifies whether the two safety signals catch different unknown cases.

Complementarity is supported when both:

- `jointenergy_only > 0`
- `mc_dropout_only > 0`

for multiple seeds.


In [10]:
# ============================================================
# 10. Complementarity
# ============================================================

overlap_df = pd.DataFrame(
    overlap_rows
)

overlap_df.to_csv(
    TABLE_DIR /
    "unknown_referral_overlap.csv",
    index=False,
)

display(
    overlap_df
)

print(
    "\nMean unknown cases referred "
    "by fixed fusion:",
    overlap_df[
        "fusion_referred"
    ].mean()
)

print(
    "Mean JointEnergy-only unique referrals:",
    overlap_df[
        "jointenergy_only"
    ].mean()
)

print(
    "Mean MC-Dropout-only unique referrals:",
    overlap_df[
        "mc_dropout_only"
    ].mean()
)


,seed,unknown_cases,jointenergy_referred,mc_dropout_referred,jointenergy_and_mc_intersection,jointenergy_only,mc_dropout_only,union_joint_or_mc,fusion_referred
0,42,32,9,6,3,6,3,12,16
1,123,32,8,7,3,5,4,12,12
2,2026,32,7,12,3,4,9,16,18



Mean unknown cases referred by fixed fusion: 15.333333333333334
Mean JointEnergy-only unique referrals: 5.0
Mean MC-Dropout-only unique referrals: 5.333333333333333


# 11. Fusion Threshold Sensitivity


In [11]:
# ============================================================
# 11. Fusion sensitivity
# ============================================================

fusion_sensitivity_df = (
    pd.DataFrame(
        fusion_sensitivity_rows
    )
)

fusion_sensitivity_df.to_csv(
    TABLE_DIR /
    "fusion_threshold_sensitivity_by_seed.csv",
    index=False,
)

display(
    fusion_sensitivity_df
)

fusion_sensitivity_aggregate = (
    fusion_sensitivity_df
    .groupby(
        "requested_val_false_referral_rate"
    )
    .agg(
        familiar_false_referral_mean=(
            "familiar_test_false_referral_rate",
            "mean",
        ),

        familiar_false_referral_sd=(
            "familiar_test_false_referral_rate",
            "std",
        ),

        normal_false_referral_mean=(
            "normal_test_false_referral_rate",
            "mean",
        ),

        known_false_referral_mean=(
            "known_only_test_false_referral_rate",
            "mean",
        ),

        unknown_referral_sensitivity_mean=(
            "unknown_referral_sensitivity",
            "mean",
        ),

        unknown_referral_sensitivity_sd=(
            "unknown_referral_sensitivity",
            "std",
        ),

        rare_to_normal_mean=(
            "post_referral_rare_to_normal_rate",
            "mean",
        ),

        strict_false_reassurance_mean=(
            "post_referral_explicit_normal_false_reassurance",
            "mean",
        ),
    )
    .reset_index()
)

fusion_sensitivity_aggregate.to_csv(
    TABLE_DIR /
    "fusion_threshold_sensitivity_mean_sd.csv",
    index=False,
)

display(
    fusion_sensitivity_aggregate
)


,seed,requested_val_false_referral_rate,threshold,actual_val_false_referral_rate,familiar_test_false_referral_rate,normal_test_false_referral_rate,known_only_test_false_referral_rate,unknown_referral_sensitivity,post_referral_rare_to_normal_rate,post_referral_explicit_normal_false_reassurance
0,42,0.05,1.207388,0.05,0.063047,0.000000,0.082380,0.28125,0.21875,0.09375
1,42,0.10,0.862129,0.10,0.103327,0.007463,0.132723,0.50000,0.15625,0.09375
2,42,0.15,0.690490,0.15,0.131349,0.014925,0.167048,0.59375,0.09375,0.09375
3,123,0.05,1.250097,0.05,0.035026,0.000000,0.045767,0.28125,0.12500,0.12500
4,123,0.10,0.955950,0.10,0.073555,0.014925,0.091533,0.37500,0.09375,0.09375
5,123,0.15,0.792874,0.15,0.105079,0.022388,0.130435,0.43750,0.09375,0.09375
6,2026,0.05,1.137529,0.05,0.054291,0.007463,0.068650,0.40625,0.18750,0.18750
7,2026,0.10,0.843880,0.10,0.106830,0.029851,0.130435,0.56250,0.15625,0.15625
8,2026,0.15,0.649915,0.15,0.152364,0.067164,0.178490,0.62500,0.12500,0.12500


,requested_val_false_referral_rate,familiar_false_referral_mean,familiar_false_referral_sd,normal_false_referral_mean,known_false_referral_mean,unknown_referral_sensitivity_mean,unknown_referral_sensitivity_sd,rare_to_normal_mean,strict_false_reassurance_mean
0,0.05,0.050788,0.014335,0.002488,0.065599,0.322917,0.072169,0.177083,0.135417
1,0.10,0.094571,0.018284,0.017413,0.118230,0.479167,0.095470,0.135417,0.114583
2,0.15,0.129597,0.023691,0.034826,0.158658,0.552083,0.100455,0.104167,0.104167


# 12. Calibration Results

ECE and Brier score are reported before and after temperature scaling.

Lower is better.


In [12]:
# ============================================================
# 12. Calibration
# ============================================================

calibration_df = pd.DataFrame(
    calibration_rows
)

calibration_df.to_csv(
    TABLE_DIR /
    "temperature_scaling_by_seed.csv",
    index=False,
)

display(
    calibration_df
)

calibration_aggregate_rows = []

for metric in [
    "temperature",
    "validation_ece_before",
    "validation_ece_after",
    "validation_brier_before",
    "validation_brier_after",
    "familiar_test_ece_before",
    "familiar_test_ece_after",
    "familiar_test_brier_before",
    "familiar_test_brier_after",
]:
    values = (
        calibration_df[
            metric
        ].astype(float)
    )

    calibration_aggregate_rows.append({
        "Metric":
            metric,

        "Mean":
            float(
                values.mean()
            ),

        "SD":
            float(
                values.std(
                    ddof=1
                )
            ),
    })

calibration_aggregate_df = pd.DataFrame(
    calibration_aggregate_rows
)

calibration_aggregate_df.to_csv(
    TABLE_DIR /
    "temperature_scaling_mean_sd.csv",
    index=False,
)

display(
    calibration_aggregate_df
)


,seed,temperature,validation_ece_before,validation_ece_after,validation_brier_before,validation_brier_after,familiar_test_ece_before,familiar_test_ece_after,familiar_test_brier_before,familiar_test_brier_after
0,42,0.422118,0.211335,0.107523,0.088506,0.065722,0.205053,0.098814,0.082115,0.058660
1,123,0.649055,0.100036,0.055502,0.048756,0.044390,0.093061,0.048404,0.044287,0.040057
2,2026,0.627235,0.127403,0.073804,0.059137,0.052643,0.121658,0.068450,0.054028,0.047208


,Metric,Mean,SD
0,temperature,0.566136,0.125199
1,validation_ece_before,0.146258,0.057996
2,validation_ece_after,0.078943,0.026389
3,validation_brier_before,0.065466,0.020617
4,validation_brier_after,0.054252,0.010757
5,familiar_test_ece_before,0.139924,0.058187
6,familiar_test_ece_after,0.071889,0.025380
7,familiar_test_brier_before,0.060143,0.019642
8,familiar_test_brier_after,0.048642,0.009384


# 13. Final Primary Safety Improvement


In [13]:
# ============================================================
# 13. Final safety summary
# ============================================================

fusion_rows = (
    method_results_df[
        method_results_df[
            "method"
        ] == "Fusion_50_50"
    ]
)

baseline_mean = float(
    fusion_rows[
        "closedset_rare_to_normal_rate"
    ].mean()
)

fusion_rtn_mean = float(
    fusion_rows[
        "post_referral_rare_to_normal_rate"
    ].mean()
)

absolute_reduction = (
    baseline_mean
    - fusion_rtn_mean
)

relative_reduction = (
    absolute_reduction
    / baseline_mean
    if baseline_mean > 0
    else np.nan
)

final_safety_summary = {
    "closedset_rare_to_normal_rate":
        baseline_mean,

    "fusion_post_referral_rare_to_normal_rate":
        fusion_rtn_mean,

    "absolute_reduction":
        absolute_reduction,

    "relative_reduction":
        relative_reduction,

    "fusion_unknown_referral_sensitivity":
        float(
            fusion_rows[
                "unknown_referral_sensitivity"
            ].mean()
        ),

    "fusion_familiar_false_referral_rate":
        float(
            fusion_rows[
                "familiar_test_false_referral_rate"
            ].mean()
        ),

    "fusion_normal_false_referral_rate":
        float(
            fusion_rows[
                "normal_test_false_referral_rate"
            ].mean()
        ),

    "fusion_known_only_false_referral_rate":
        float(
            fusion_rows[
                "known_only_test_false_referral_rate"
            ].mean()
        ),

    "fusion_unknown_auroc":
        float(
            fusion_rows[
                "unknown_auroc"
            ].mean()
        ),

    "fusion_unknown_aupr":
        float(
            fusion_rows[
                "unknown_aupr"
            ].mean()
        ),
}

print(
    json.dumps(
        final_safety_summary,
        indent=2,
    )
)

with open(
    TABLE_DIR /
    "final_safefundus_safety_summary.json",
    "w",
) as f:
    json.dump(
        final_safety_summary,
        f,
        indent=2,
    )


{
  "closedset_rare_to_normal_rate": 0.2708333333333333,
  "fusion_post_referral_rare_to_normal_rate": 0.13541666666666666,
  "absolute_reduction": 0.13541666666666666,
  "relative_reduction": 0.5,
  "fusion_unknown_referral_sensitivity": 0.4791666666666667,
  "fusion_familiar_false_referral_rate": 0.09457092819614711,
  "fusion_normal_false_referral_rate": 0.017412935323383085,
  "fusion_known_only_false_referral_rate": 0.11823035850495804,
  "fusion_unknown_auroc": 0.7821438995913601,
  "fusion_unknown_aupr": 0.21918033360616787
}


# 14. Save Final Method Metadata


In [14]:
# ============================================================
# 14. Metadata
# ============================================================

metadata = {
    "project":
        "SafeFundus",

    "classifier":
        "Swin-Tiny + ASL + explicit NORMAL output",

    "open_set_component_1":
        "Normal-Aware Dual-Mode JointEnergy",

    "open_set_component_2":
        "20-pass prediction-head MC Dropout variance",

    "jointenergy_final_normalization":
        "z-score on complete familiar validation set after dual-mode score",

    "mc_dropout_normalization":
        "z-score on familiar validation set from Notebook 08",

    "fusion":
        "R = 0.5 * J_N + 0.5 * U_N",

    "fusion_weight_tuned_on_test":
        False,

    "primary_referral_threshold":
        "90th percentile of familiar validation fusion score",

    "primary_validation_false_referral_target":
        CONFIG[
            "primary_val_false_referral_rate"
        ],

    "temperature_scaling":
        "single scalar T per seed fitted on familiar validation logits",

    "temperature_affects_open_set_score":
        False,

    "unknown_test_used_for_tuning":
        False,
}

with open(
    OUTPUT_DIR /
    "final_method_metadata.json",
    "w",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

print(
    json.dumps(
        metadata,
        indent=2,
    )
)


{
  "project": "SafeFundus",
  "classifier": "Swin-Tiny + ASL + explicit NORMAL output",
  "open_set_component_1": "Normal-Aware Dual-Mode JointEnergy",
  "open_set_component_2": "20-pass prediction-head MC Dropout variance",
  "jointenergy_final_normalization": "z-score on complete familiar validation set after dual-mode score",
  "mc_dropout_normalization": "z-score on familiar validation set from Notebook 08",
  "fusion": "R = 0.5 * J_N + 0.5 * U_N",
  "fusion_weight_tuned_on_test": false,
  "primary_referral_threshold": "90th percentile of familiar validation fusion score",
  "primary_validation_false_referral_target": 0.1,
  "temperature_scaling": "single scalar T per seed fitted on familiar validation logits",
  "temperature_affects_open_set_score": false,
  "unknown_test_used_for_tuning": false
}


# 15. Create One Complete Output ZIP

The notebook automatically creates:

**`09_SafeFundus_Fusion_Calibration_Ablation.zip`**

The ZIP contains:

- per-image final JointEnergy, MC Dropout, and fusion scores;
- per-seed fusion thresholds;
- calibration temperatures;
- calibrated probability thresholds;
- classifier / JointEnergy / MC Dropout / fusion ablation;
- unknown referral overlap;
- 5% / 10% / 15% fusion sensitivity analysis;
- unknown AUROC / AUPR / FPR@95TPR;
- primary rare-to-normal safety result;
- ECE and Brier score before/after calibration;
- figures;
- final method metadata.


In [15]:
# ============================================================
# 15. Complete ZIP
# ============================================================

ZIP_PATH = (
    WORK_ROOT /
    f"{NOTEBOOK_STEM}.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(
        ZIP_PATH.with_suffix("")
    ),
    format="zip",
    root_dir=OUTPUT_DIR,
)

size_mb = (
    ZIP_PATH.stat().st_size
    / (1024 ** 2)
)

print("=" * 80)
print("COMPLETE OUTPUT ZIP CREATED")
print("=" * 80)
print(ZIP_PATH)

print(
    f"ZIP size: "
    f"{size_mb:.2f} MB"
)


COMPLETE OUTPUT ZIP CREATED
/kaggle/working/09_SafeFundus_Fusion_Calibration_Ablation.zip
ZIP size: 0.25 MB


# After Notebook 09

Upload:

**`09_SafeFundus_Fusion_Calibration_Ablation.zip`**

I will verify the final fusion result and freeze the internal RFMiD SafeFundus method.

After that, the remaining major experimental stage is **external validation on RFMiD 2.0**, using the already-frozen model, calibration, fusion weights, and referral policy without tuning on the external dataset.
